# 📊 04 — Training Curves & Evaluation

Analyze training results, plot loss curves, and compute BLEU scores.

In [ ]:
import sys
sys.path.insert(0, '..')
import json
import matplotlib.pyplot as plt
import numpy as np
from training.metrics import compute_bleu, print_bleu_report

## 1. Load Training History

In [ ]:
history_path = '../checkpoints/training_history.json'

try:
    with open(history_path, 'r') as f:
        history = json.load(f)
    print(f'Loaded {len(history["train_loss"])} epochs of training history')
except FileNotFoundError:
    print('No training history found. Generating sample data for demo...')
    epochs = 30
    history = {
        'train_loss': [5.0 * np.exp(-0.1 * i) + 0.5 + np.random.normal(0, 0.1) for i in range(epochs)],
        'val_loss': [5.2 * np.exp(-0.08 * i) + 0.8 + np.random.normal(0, 0.15) for i in range(epochs)],
        'train_acc': [min(95, 10 + 3*i + np.random.normal(0, 2)) for i in range(epochs)],
        'val_acc': [min(85, 8 + 2.5*i + np.random.normal(0, 3)) for i in range(epochs)],
    }

## 2. Loss Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss curve
axes[0].plot(history['train_loss'], label='Train Loss', color='#2196F3', lw=2)
axes[0].plot(history['val_loss'], label='Val Loss', color='#E91E63', lw=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training & Validation Loss', fontsize=14)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Accuracy curve
axes[1].plot(history['train_acc'], label='Train Acc', color='#4CAF50', lw=2)
axes[1].plot(history['val_acc'], label='Val Acc', color='#FF9800', lw=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy (%)', fontsize=12)
axes[1].set_title('Training & Validation Accuracy', fontsize=14)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.suptitle('CSLT Model Training Progress', fontsize=16, fontweight='bold')
plt.tight_layout(); plt.show()

print(f'Best train loss: {min(history["train_loss"]):.4f}')
print(f'Best val loss  : {min(history["val_loss"]):.4f}')

## 3. Overfitting Analysis

In [ ]:
gap = [v - t for t, v in zip(history['train_loss'], history['val_loss'])]

fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(range(len(gap)), gap, alpha=0.3, color='#E91E63')
ax.plot(gap, color='#E91E63', lw=2)
ax.axhline(0, color='gray', ls='--')
ax.set_xlabel('Epoch'); ax.set_ylabel('Val Loss - Train Loss')
ax.set_title('Overfitting Gap (positive = overfitting)', fontsize=13)
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

if gap[-1] > 1.0:
    print('⚠️  Model is overfitting — consider more dropout or data augmentation')
else:
    print('✅ Gap is reasonable — no severe overfitting detected')

## 4. BLEU Score Evaluation

In [ ]:
# Example translations (replace with real model output after training)
predictions = [
    'the girl is walking to school',
    'he is eating food',
    'she wants to go home',
    'the boy is playing basketball',
    'thank you very much',
]
references = [
    'a girl walks to school',
    'he is eating his food',
    'she wants to go home now',
    'the boy plays basketball',
    'thank you so much',
]

scores = print_bleu_report(predictions, references)

## 5. BLEU Score Interpretation

In [ ]:
categories = ['BLEU-1', 'BLEU-2', 'BLEU-3', 'BLEU-4']
values = [scores['bleu1'], scores['bleu2'], scores['bleu3'], scores['bleu4']]
colors = ['#4CAF50', '#2196F3', '#FF9800', '#E91E63']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(categories, values, color=colors, edgecolor='white', width=0.6)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.1f}', ha='center', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12)
ax.set_title('BLEU Score Results', fontsize=14, fontweight='bold')
ax.set_ylim(0, 100)
ax.axhline(25, color='gray', ls='--', alpha=0.5, label='SOTA target (~25)')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

print('\n✅ Evaluation complete!')